# *Deep Learning Basics with PyTorch*
# Part I — Foundations of Machine Learning
## Chapter 3: Basic Learning Behavior

This chapter introduces simple supervised learning behavior with tiny beginner-friendly datasets.
The goal is to see how models fit patterns, how train/test evaluation works, and how model flexibility can be too low or too high.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'retina'


## Simple Regression Example

We begin with a tiny synthetic regression problem and fit a straight line to the data.


In [ ]:
x = np.array([0., 1., 2., 3., 4., 5.])
y = np.array([0.8, 2.1, 3.3, 4.7, 5.8, 7.2])

slope, intercept = np.polyfit(x, y, deg=1)
y_pred = slope * x + intercept
mse = np.mean((y - y_pred) ** 2)

print(f"slope: {slope:.3f}")
print(f"intercept: {intercept:.3f}")
print(f"MSE: {mse:.3f}")

for xi, yi, pi in zip(x, y, y_pred):
    print(f"x={xi:.1f} | y={yi:.2f} | pred={pi:.2f}")

plt.figure(figsize=(4.5, 3.2))
plt.scatter(x, y, label="data")
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, slope * xx + intercept, color="red", label="fit")
plt.xlabel("input")
plt.ylabel("target")
plt.legend()
plt.tight_layout()
plt.show()


## Simple Classification Example

Next we use a tiny flower dataset and a simple petal-length threshold to separate two classes.


In [ ]:
df_cls = pd.DataFrame({
    "petal_length_cm": [1.4, 1.4, 1.5, 1.3, 1.7, 1.5, 4.5, 4.5, 4.7, 3.9, 4.7, 4.4],
    "petal_width_cm": [0.2, 0.2, 0.2, 0.2, 0.4, 0.3, 1.5, 1.5, 1.4, 1.1, 1.5, 1.3],
    "label": [
        "setosa", "setosa", "setosa", "setosa", "setosa", "setosa",
        "versicolor", "versicolor", "versicolor", "versicolor", "versicolor", "versicolor"
    ]
})

y_cls = (df_cls["label"] == "versicolor").astype(int).values
threshold = 0.5 * (
    df_cls.loc[y_cls == 0, "petal_length_cm"].mean()
    + df_cls.loc[y_cls == 1, "petal_length_cm"].mean()
)
pred_cls = (df_cls["petal_length_cm"] > threshold).astype(int).values
accuracy = np.mean(pred_cls == y_cls)

print(f"petal-length threshold: {threshold:.2f} cm")
print(f"accuracy: {accuracy:.2f}")

plt.figure(figsize=(4.5, 3.2))
for label_name, color in [("setosa", "tab:blue"), ("versicolor", "tab:orange")]:
    subset = df_cls[df_cls["label"] == label_name]
    plt.scatter(
        subset["petal_length_cm"],
        subset["petal_width_cm"],
        color=color,
        label=label_name,
        s=50,
        alpha=0.8,
    )
plt.axvline(threshold, color="black", linestyle="--", label="threshold")
plt.xlabel("petal length (cm)")
plt.ylabel("petal width (cm)")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## Train/Test Evaluation Intuition

A model may fit the training data well, but we also care about how it performs on unseen test data.


In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0, 6, 16)
y = 1.2 * x + 0.8 + rng.normal(0, 0.6, size=len(x))

indices = rng.permutation(len(x))
train_idx = indices[:12]
test_idx = indices[12:]

x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

slope, intercept = np.polyfit(x_train, y_train, deg=1)
train_pred = slope * x_train + intercept
test_pred = slope * x_test + intercept

train_mse = np.mean((y_train - train_pred) ** 2)
test_mse = np.mean((y_test - test_pred) ** 2)

print("train size:", len(x_train))
print("test size:", len(x_test))
print(f"train MSE: {train_mse:.3f}")
print(f"test MSE: {test_mse:.3f}")

plt.figure(figsize=(4.8, 3.4))
plt.scatter(x_train, y_train, label="train")
plt.scatter(x_test, y_test, label="test")
xx = np.linspace(x.min(), x.max(), 100)
plt.plot(xx, slope * xx + intercept, color="red", label="fit")
plt.xlabel("input")
plt.ylabel("target")
plt.legend()
plt.tight_layout()
plt.show()


## Underfitting vs Overfitting

Changing model flexibility affects both training error and test error. Too simple can underfit, while too flexible can overfit.


In [ ]:
rng = np.random.default_rng(2)
x = np.linspace(-3, 3, 12)
y = 0.5 * x**2 + x + rng.normal(0, 0.8, size=len(x))

indices = rng.permutation(len(x))
train_idx = indices[:8]
test_idx = indices[8:]

x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

degrees = [1, 2, 5]
rows = []
xx = np.linspace(x.min(), x.max(), 300)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)

for i, degree in enumerate(degrees):
    coeffs = np.polyfit(x_train, y_train, deg=degree)
    train_pred = np.polyval(coeffs, x_train)
    test_pred = np.polyval(coeffs, x_test)
    rows.append({
        "degree": degree,
        "train_MSE": np.mean((y_train - train_pred) ** 2),
        "test_MSE": np.mean((y_test - test_pred) ** 2),
    })
    ax[i].scatter(x_train, y_train, label="train")
    ax[i].scatter(x_test, y_test, label="test")
    ax[i].plot(xx, np.polyval(coeffs, xx), color="red")
    ax[i].set_title(f"degree {degree}")
    ax[i].set_xlabel("input")

ax[0].set_ylabel("target")
ax[0].legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame(rows).round(3)


## Chapter 3 Summary

- Regression predicts continuous values, while classification predicts categories or labels.
- A model is useful only if it performs reasonably on data it did not see during fitting.
- Train/test metrics help us check whether a pattern generalizes beyond the training set.
- Model flexibility matters: too little can underfit and too much can overfit.
